In [ ]:

# Section 3: Deep Learning and Advanced Models



# Reproducibility + stdlib

import os
import random
import time
import shutil
import numpy as np

random.seed(42)
np.random.seed(42)
os.environ["PYTHONHASHSEED"] = "42"


# Core libraries

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix


# TensorFlow / Keras

import tensorflow as tf
tf.random.set_seed(42)

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, BatchNormalization, Embedding,
    SimpleRNN, LSTM, GRU, Conv1D, MaxPooling1D,
    GlobalAveragePooling2D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50


# PyTorch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


# Plotting

import matplotlib.pyplot as plt
import seaborn as sns


# YOLO (guarded import)

try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
except Exception as e:
    YOLO_AVAILABLE = False
    print("YOLO unavailable, skipping detection later. Reason:", str(e))



# Section 3.0: Load and prepare review data

df = pd.read_csv("amazon_reviews_us_Apparel_v1_00.csv", nrows=50000)
df.dropna(subset=["review_body", "star_rating"], inplace=True)
df = df[df["star_rating"].isin([1, 2, 4, 5])]
df["review_body"] = df["review_body"].str.lower().str.replace(r"[^a-z0-9\s]", "", regex=True)

# TF-IDF features
vectorizer = TfidfVectorizer(max_features=1000, stop_words="english")
X = vectorizer.fit_transform(df["review_body"]).toarray()

# Labels for Keras (one hot) and for PyTorch (integer)
encoder = OneHotEncoder(sparse_output=False)
y_keras = encoder.fit_transform(df["star_rating"].values.reshape(-1, 1))
label_encoder = LabelEncoder()
y_pytorch = label_encoder.fit_transform(df["star_rating"].values)

# Use 1D labels for stratify
y_labels = df["star_rating"].values

# Split once for Keras and once for PyTorch
X_train, X_test, y_train, y_test = train_test_split(
    X, y_keras, test_size=0.2, random_state=42, stratify=y_labels
)
X_train_pt, X_test_pt, y_train_pt, y_test_pt = train_test_split(
    X, y_pytorch, test_size=0.2, random_state=42, stratify=y_pytorch
)



# Section 3.1: Single Layer Perceptron (Keras)

print("\n=== Section 3.1: Single-Layer Perceptron (Keras) ===")
perceptron = Sequential([Dense(4, activation="softmax", input_shape=(X_train.shape[1],))])
perceptron.compile(optimizer=Adam(learning_rate=0.01), loss="categorical_crossentropy", metrics=["accuracy"])
perceptron.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=1)

y_pred = perceptron.predict(X_test).argmax(axis=1)
y_true = y_test.argmax(axis=1)
print(classification_report(y_true, y_pred, target_names=encoder.categories_[0].astype(str)))



# Section 3.2: Deep Neural Network with regularization (Keras)

print("\n=== Section 3.2: DNN with Dropout and BatchNorm (Keras) ===")
model_dnn = Sequential([
    Dense(256, activation="relu", input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.2),
    Dense(64, activation="relu"),
    Dropout(0.1),
    Dense(4, activation="softmax")
])
model_dnn.compile(optimizer=Adam(0.001), loss="categorical_crossentropy", metrics=["accuracy"])

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
model_dnn.fit(X_train, y_train, validation_split=0.1, epochs=15, batch_size=64, callbacks=[early_stop], verbose=1)

y_pred_dnn = model_dnn.predict(X_test).argmax(axis=1)
print(classification_report(y_true, y_pred_dnn, target_names=encoder.categories_[0].astype(str)))



# Section 3.3: PyTorch Classification DNN

print("\n=== Section 3.3: PyTorch Classification DNN ===")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ClassificationNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(1000, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 4)
        )
    def forward(self, x):
        return self.model(x)

X_train_tensor = torch.tensor(X_train_pt, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_pt, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_pt, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_pt, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

model_pt = ClassificationNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_pt.parameters(), lr=0.001)

for epoch in range(5):
    model_pt.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model_pt(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

model_pt.eval()
with torch.no_grad():
    logits = model_pt(X_test_tensor.to(device))
    y_pred_classes = torch.argmax(logits, axis=1).cpu().numpy()

print(classification_report(y_test_tensor.numpy(), y_pred_classes, target_names=label_encoder.classes_.astype(str)))



# Section 3.4: Keras vs PyTorch efficiency comparison

print("\n=== Section 3.4: Keras vs PyTorch Comparison ===")
# Keras timing
start_time_keras = time.time()
model_dnn.fit(X_train, y_train, validation_split=0.1, epochs=5, batch_size=64, verbose=0)
keras_train_time = time.time() - start_time_keras
y_pred_keras = model_dnn.predict(X_test).argmax(axis=1)
y_true_keras = y_test.argmax(axis=1)
keras_acc = accuracy_score(y_true_keras, y_pred_keras)

# PyTorch timing
X_train_tensor2 = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor2 = torch.tensor(np.argmax(y_train, axis=1), dtype=torch.long).to(device)
train_dataset2 = TensorDataset(X_train_tensor2, y_train_tensor2)
train_loader2 = DataLoader(train_dataset2, batch_size=64, shuffle=True)

model_pt2 = ClassificationNN().to(device)
optimizer2 = optim.Adam(model_pt2.parameters(), lr=0.001)

start_time_pt = time.time()
for epoch in range(5):
    model_pt2.train()
    for xb, yb in train_loader2:
        pred = model_pt2(xb)
        loss = loss_fn(pred, yb)
        optimizer2.zero_grad()
        loss.backward()
        optimizer2.step()
pt_train_time = time.time() - start_time_pt

model_pt2.eval()
with torch.no_grad():
    y_pred_pt = model_pt2(torch.tensor(X_test, dtype=torch.float32).to(device)).argmax(dim=1).cpu().numpy()
pt_acc = accuracy_score(y_true_keras, y_pred_pt)

print(f"Keras Accuracy:   {keras_acc:.4f} | Training Time: {keras_train_time:.2f} seconds")
print(f"PyTorch Accuracy: {pt_acc:.4f} | Training Time: {pt_train_time:.2f} seconds")



# Section 3.5: Sequential text models (RNN, LSTM, GRU, CRNN)

print("\n=== Section 3.5: Sequential Text Models (RNN, LSTM, GRU, CRNN) ===")

df_seq = df.copy()
df_seq["sentiment"] = df_seq["star_rating"].apply(lambda x: 1 if x >= 4 else 0)

tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(df_seq["review_body"])
seqs = tokenizer.texts_to_sequences(df_seq["review_body"])
padded = pad_sequences(seqs, maxlen=200, padding="post", truncating="post")

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    padded, df_seq["sentiment"].values, test_size=0.2, random_state=42, stratify=df_seq["sentiment"]
)

def train_eval_keras(model, name):
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(X_train_s, y_train_s, validation_split=0.1, epochs=5, batch_size=64, verbose=1)
    y_hat = (model.predict(X_test_s) > 0.5).astype("int32")
    print(f"\nClassification Report ({name}):")
    print(classification_report(y_test_s, y_hat))

# Simple RNN
model_rnn = Sequential([Embedding(10000, 64, input_length=200), SimpleRNN(64), Dense(1, activation="sigmoid")])
train_eval_keras(model_rnn, "RNN Sentiment Classifier")

# LSTM
model_lstm = Sequential([Embedding(10000, 64, input_length=200), LSTM(64), Dense(1, activation="sigmoid")])
train_eval_keras(model_lstm, "LSTM Sentiment Classifier")

# GRU
model_gru = Sequential([Embedding(10000, 64, input_length=200), GRU(64), Dense(1, activation="sigmoid")])
train_eval_keras(model_gru, "GRU Sentiment Classifier")

# CRNN (Conv1D + LSTM)
model_crnn = Sequential([
    Embedding(10000, 64, input_length=200),
    Conv1D(64, 5, activation="relu"),
    MaxPooling1D(2),
    LSTM(64),
    Dense(1, activation="sigmoid")
])
train_eval_keras(model_crnn, "CRNN Sentiment Classifier")



# Section 3.7: Fix images layout + ResNet50 fine-tuning

print("\n=== Section 3.7: ResNet50 Fine Tuning (folder based) ===")

#Normalize folder structure 
root = "dataset/images"            
splits = ["train", "val", "test"]
valid_ext = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# Move accidentally nested splits (e.g., dataset/images/train/val -> dataset/images/val)
for split in splits:
    sdir = os.path.join(root, split)
    if not os.path.isdir(sdir):
        continue
    for other in splits:
        if other == split:
            continue
        nested = os.path.join(sdir, other)
        if os.path.isdir(nested):
            target = os.path.join(root, other)
            os.makedirs(target, exist_ok=True)
            for name in os.listdir(nested):
                shutil.move(os.path.join(nested, name), os.path.join(target, name))
            try:
                os.rmdir(nested)
            except OSError:
                pass
            print(f"Moved nested '{nested}' into '{target}'")

# Ensure each split has at least one class folder; move loose images into 'unknown'
for split in splits:
    sdir = os.path.join(root, split)
    if not os.path.isdir(sdir):
        print(f"Warning: missing folder: {sdir}")
        continue
    loose = [f for f in os.listdir(sdir)
             if os.path.isfile(os.path.join(sdir, f))
             and f.lower().endswith(valid_ext)]
    if loose:
        unk = os.path.join(sdir, "unknown")
        os.makedirs(unk, exist_ok=True)
        for f in loose:
            shutil.move(os.path.join(sdir, f), os.path.join(unk, f))
        print(f"Moved {len(loose)} images into {unk}")

    classes = [d for d in os.listdir(sdir) if os.path.isdir(os.path.join(sdir, d))]
    if not classes:
        print(f"Warning: {sdir} has no class subfolders. "
              "Expected structure: dataset/images/{{train,val,test}}/{{class}}/img.jpg")

# Generators
image_size = (128, 128)
batch_size = 32

train_aug = ImageDataGenerator(rescale=1./255, rotation_range=25, zoom_range=0.2, horizontal_flip=True)
val_aug   = ImageDataGenerator(rescale=1./255)
test_aug  = ImageDataGenerator(rescale=1./255)

train_gen = train_aug.flow_from_directory(
    os.path.join(root, "train"),
    target_size=image_size,
    batch_size=batch_size,
    class_mode="categorical"
)
val_gen = val_aug.flow_from_directory(
    os.path.join(root, "val"),
    target_size=image_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)
test_gen = test_aug.flow_from_directory(
    os.path.join(root, "test"),
    target_size=image_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)

print(f"train: {train_gen.samples} images | classes: {train_gen.class_indices}")
print(f"val:   {val_gen.samples} images | classes: {val_gen.class_indices}")
print(f"test:  {test_gen.samples} images | classes: {test_gen.class_indices}")

# Hard stop if still empty
if train_gen.samples == 0 or val_gen.samples == 0:
    raise ValueError(
        "No images found for training/validation.\n"
        "Ensure files are under dataset/images/{train,val}/{class_name}/image.jpg"
    )

#3) Build & fine-tune ResNet50 
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(image_size[0], image_size[1], 3))
# Unfreeze last 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers[-30:]:
    layer.trainable = True

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(train_gen.num_classes, activation='softmax')(x)

cnn_model = Model(inputs=base_model.input, outputs=predictions)
cnn_model.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

fine_tune_cb = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history = cnn_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=[fine_tune_cb],
    verbose=1
)

#4) Evaluate 
test_gen.reset()
y_true_img = test_gen.classes
class_labels = list(test_gen.class_indices.keys())
y_pred_img = cnn_model.predict(test_gen)
y_pred_classes_img = np.argmax(y_pred_img, axis=1)

print("\nClassification Report (ResNet50 fine-tuned):")
print(classification_report(y_true_img, y_pred_classes_img, target_names=class_labels))

cm = confusion_matrix(y_true_img, y_pred_classes_img)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_labels, yticklabels=class_labels, cmap='Blues')
plt.title("Confusion Matrix - ResNet50")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()



# Section 3.8: YOLOv8 object detection

from pathlib import Path
# - YOLO check & conditional training -
from pathlib import Path

def count_txt(base="dataset"):
    return sum(1 for _ in Path(base, "labels").rglob("*.txt"))

num_label_files = count_txt("dataset")
print(f"YOLO: found {num_label_files} label files under dataset/labels/**")

if num_label_files == 0:
    print("Skipping YOLO — no .txt labels found. See https://docs.ultralytics.com/datasets/detect")
else:
    yolo = YOLO("yolov8n.pt")
    yolo.train(
        data="yolo_dataset.yaml",
        epochs=10,
        imgsz=640,
        batch=16,
        name="capstone_yolo_model"
    )
    yolo.val()
    yolo.export(format="tflite")


base = Path("dataset")
for split in ["train","val","test"]:
    imgs = list((base/"images"/split).glob("*.*"))
    lbls = list((base/"labels"/split).glob("*.txt"))
    print(f"{split}: {len(imgs)} images, {len(lbls)} labels")


print("\n=== Section 3.8: YOLOv8 Object Detection ===")
if YOLO_AVAILABLE:
    try:
        yolo = YOLO("yolov8n.pt")
        yolo.train(
            data="yolo_dataset.yaml",
            epochs=10,
            imgsz=640,
            batch=16,
            name="capstone_yolo_model"
        )
        yolo.val()
        yolo.export(format="tflite")
    except Exception as e:
        print("YOLO training skipped. Reason:", str(e))
else:
    print("YOLO skipped (ultralytics not installed).")


AttributeError: `np.complex_` was removed in the NumPy 2.0 release. Use `np.complex128` instead.